# Notebook title

description here

### Import Libraries

fill this out later

In [1]:
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import scheduleleaguev2

pd.set_option('display.max_columns', None)

SEASON = "2025-26"
DATA = "../data/2025_raw_box_score_team_stats.csv"


### Consolidate games 

fill this out later

In [2]:
def game_consolidation(season_data):
    '''
    Fill this out later
    '''
    df = pd.read_csv(season_data)
    df = df.drop(columns=[
                "TEAM_ID", "SEASON_ID", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
                "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
            ])

    home = df[df["MATCHUP"].str.contains("vs.")]
    away = df[df["MATCHUP"].str.contains("@")]
    games = home.merge(away, on="GAME_ID", suffixes=("_HOME", "_AWAY"))
    games = games.rename(columns={"MATCHUP_AWAY": 'MATCHUP'})
    games['WINNER'] = np.where(games['WL_HOME'] == "W", games['TEAM_NAME_HOME'], games['TEAM_NAME_AWAY'])

    games = games[[
            "GAME_ID", "GAME_DATE_HOME", "MATCHUP", 
            "TEAM_ABBREVIATION_AWAY", "TEAM_NAME_AWAY", "PTS_AWAY",
            "TEAM_ABBREVIATION_HOME", "TEAM_NAME_HOME", "PTS_HOME", "WINNER"
        ]]

    return games


test = game_consolidation(DATA)
test


,GAME_ID,GAME_DATE_HOME,MATCHUP,TEAM_ABBREVIATION_AWAY,TEAM_NAME_AWAY,PTS_AWAY,TEAM_ABBREVIATION_HOME,TEAM_NAME_HOME,PTS_HOME,WINNER
0,22500001,2025-10-21,HOU @ OKC,HOU,Houston Rockets,124,OKC,Oklahoma City Thunder,125,Oklahoma City Thunder
1,22500002,2025-10-21,GSW @ LAL,GSW,Golden State Warriors,119,LAL,Los Angeles Lakers,109,Golden State Warriors
2,22500086,2025-10-22,WAS @ MIL,WAS,Washington Wizards,120,MIL,Milwaukee Bucks,133,Milwaukee Bucks
3,22500003,2025-10-22,CLE @ NYK,CLE,Cleveland Cavaliers,111,NYK,New York Knicks,119,New York Knicks
4,22500081,2025-10-22,MIA @ ORL,MIA,Miami Heat,121,ORL,Orlando Magic,125,Orlando Magic
...,...,...,...,...,...,...,...,...,...,...
1220,22501194,2026-04-12,MEM @ HOU,MEM,Memphis Grizzlies,101,HOU,Houston Rockets,132,Houston Rockets
1221,22501199,2026-04-12,GSW @ LAC,GSW,Golden State Warriors,110,LAC,LA Clippers,115,LA Clippers
1222,22501198,2026-04-12,UTA @ LAL,UTA,Utah Jazz,107,LAL,Los Angeles Lakers,131,Los Angeles Lakers
1223,22501189,2026-04-12,ATL @ MIA,ATL,Atlanta Hawks,117,MIA,Miami Heat,143,Miami Heat


### Get the game start times

fill this out later


In [3]:
def nba_game_start_times(year):
    '''
    Fill this out later
    '''
    sched = scheduleleaguev2.ScheduleLeagueV2(season=year)
    df = sched.season_games.get_data_frame()

    df = df[["gameId", "gameDate", "gameDateTimeUTC","gameDateTimeEst",
            "gameStatusText", "homeTeam_teamTricode", "awayTeam_teamTricode", "gameLabel"]]

    df = df.rename(columns={"gameId": 'GAME_ID'})
    df["GAME_ID"] = pd.to_numeric(df["GAME_ID"], errors="coerce").astype("Int64")
    games_to_keep = ['', 'Emirates NBA Cup', 'NBA Mexico City Game', 'NBA Berlin Game', 'NBA London Game', 'AWS NBA Rivals Week','NBA Pioneers Classic']
    df_filtered = df.loc[df['gameLabel'].isin(games_to_keep)]

    return df_filtered

test2 = nba_game_start_times(SEASON)
test2


,GAME_ID,gameDate,gameDateTimeUTC,gameDateTimeEst,gameStatusText,homeTeam_teamTricode,awayTeam_teamTricode,gameLabel
71,22500001,10/21/2025 00:00:00,2025-10-21T23:30:00Z,2025-10-21T19:30:00Z,Final/OT2,OKC,HOU,
72,22500002,10/21/2025 00:00:00,2025-10-22T02:00:00Z,2025-10-21T22:00:00Z,Final,LAL,GSW,
73,22500003,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,NYK,CLE,
74,22500004,10/22/2025 00:00:00,2025-10-23T01:30:00Z,2025-10-22T21:30:00Z,Final,DAL,SAS,
75,22500080,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,CHA,BKN,
...,...,...,...,...,...,...,...,...
1304,22501196,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,OKC,PHX,
1305,22501197,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,SAS,DEN,
1306,22501198,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAL,UTA,
1307,22501199,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAC,GSW,


### Merge DF's and create columns with Kalshi info

fill this out later

In [4]:

def games_with_kalshi_info_converter(df1, df2):
    '''
    Fill this out later
    '''
    # merge the two dfs on GAME_ID using inner
    merged = pd.merge(df1, df2, on="GAME_ID", how="inner")
    
    # rename columns for clarity
    merged = merged.rename(columns={
        "GAME_DATE_HOME": "GAME_DATE",
        "TEAM_ABBREVIATION_AWAY": "ABBR_AWAY",
        "TEAM_NAME_AWAY": "NAME_AWAY",
        "TEAM_ABBREVIATION_HOME": "ABBR_HOME",
        "TEAM_NAME_HOME": "NAME_HOME",
        "gameStatusText": "STATUS"
    })

    # convert gameDateTimeEst column from object to datetieme
    merged["gameDateTimeEst"] = pd.to_datetime(merged["gameDateTimeEst"])

    # convert object columns to strings
    merged["ABBR_AWAY"] = merged["ABBR_AWAY"].astype(str)
    merged["ABBR_HOME"] = merged["ABBR_HOME"].astype(str)

    # create new column for clarity using 12 hour clock
    merged["GAME_START(EST)"] = (merged["gameDateTimeEst"].dt.strftime("%I:%M%p").str.lstrip("0"))

    # create Kalshi market ID's for away teams
    merged['KALSHI_ID_AWAY'] = ('KXNBAGAME-' +
        merged['gameDateTimeEst'].dt.strftime('%y') +
        merged['gameDateTimeEst'].dt.strftime('%b').str.upper() +
        merged['gameDateTimeEst'].dt.strftime('%d') + 
        merged['ABBR_AWAY'] +
        merged['ABBR_HOME'] + '-' +
        merged['ABBR_AWAY']
    )

    # create Kalshi market ID's for home teams
    merged['KALSHI_ID_HOME'] = ('KXNBAGAME-' + 
        merged['gameDateTimeEst'].dt.strftime('%y') +
        merged['gameDateTimeEst'].dt.strftime('%b').str.upper() +
        merged['gameDateTimeEst'].dt.strftime('%d') + 
        merged['ABBR_AWAY'] +
        merged['ABBR_HOME'] + '-' +
        merged['ABBR_HOME']
    )

    # create a timestamp column that will be used for fetching Kalshi data later
    merged['TIMESTAMP'] = (merged['gameDateTimeEst'].astype('int64') // 1_000_000_000)

    # reorganize final column order
    merged = merged[[
        "GAME_ID", "GAME_DATE", "GAME_START(EST)", "TIMESTAMP", 
        "KALSHI_ID_AWAY", "KALSHI_ID_HOME", "MATCHUP", 
        "ABBR_AWAY", "NAME_AWAY", "PTS_AWAY", 
        "ABBR_HOME", "NAME_HOME", "PTS_HOME",
        "WINNER", "STATUS"
    ]]

    return merged


test3 = games_with_kalshi_info_converter(test, test2)
test3

,GAME_ID,GAME_DATE,GAME_START(EST),TIMESTAMP,KALSHI_ID_AWAY,KALSHI_ID_HOME,MATCHUP,ABBR_AWAY,NAME_AWAY,PTS_AWAY,ABBR_HOME,NAME_HOME,PTS_HOME,WINNER,STATUS
0,22500001,2025-10-21,7:30PM,1761075000,KXNBAGAME-25OCT21HOUOKC-HOU,KXNBAGAME-25OCT21HOUOKC-OKC,HOU @ OKC,HOU,Houston Rockets,124,OKC,Oklahoma City Thunder,125,Oklahoma City Thunder,Final/OT2
1,22500002,2025-10-21,10:00PM,1761084000,KXNBAGAME-25OCT21GSWLAL-GSW,KXNBAGAME-25OCT21GSWLAL-LAL,GSW @ LAL,GSW,Golden State Warriors,119,LAL,Los Angeles Lakers,109,Golden State Warriors,Final
2,22500086,2025-10-22,8:00PM,1761163200,KXNBAGAME-25OCT22WASMIL-WAS,KXNBAGAME-25OCT22WASMIL-MIL,WAS @ MIL,WAS,Washington Wizards,120,MIL,Milwaukee Bucks,133,Milwaukee Bucks,Final
3,22500003,2025-10-22,7:00PM,1761159600,KXNBAGAME-25OCT22CLENYK-CLE,KXNBAGAME-25OCT22CLENYK-NYK,CLE @ NYK,CLE,Cleveland Cavaliers,111,NYK,New York Knicks,119,New York Knicks,Final
4,22500081,2025-10-22,7:00PM,1761159600,KXNBAGAME-25OCT22MIAORL-MIA,KXNBAGAME-25OCT22MIAORL-ORL,MIA @ ORL,MIA,Miami Heat,121,ORL,Orlando Magic,125,Orlando Magic,Final
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1220,22501194,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12MEMHOU-MEM,KXNBAGAME-26APR12MEMHOU-HOU,MEM @ HOU,MEM,Memphis Grizzlies,101,HOU,Houston Rockets,132,Houston Rockets,Final
1221,22501199,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12GSWLAC-GSW,KXNBAGAME-26APR12GSWLAC-LAC,GSW @ LAC,GSW,Golden State Warriors,110,LAC,LA Clippers,115,LA Clippers,Final
1222,22501198,2026-04-12,8:30PM,1776025800,KXNBAGAME-26APR12UTALAL-UTA,KXNBAGAME-26APR12UTALAL-LAL,UTA @ LAL,UTA,Utah Jazz,107,LAL,Los Angeles Lakers,131,Los Angeles Lakers,Final
1223,22501189,2026-04-12,6:00PM,1776016800,KXNBAGAME-26APR12ATLMIA-ATL,KXNBAGAME-26APR12ATLMIA-MIA,ATL @ MIA,ATL,Atlanta Hawks,117,MIA,Miami Heat,143,Miami Heat,Final


In [5]:
test3.to_csv("../data/2025_all_games_with_kalshi_info.csv", index=False)


### Fetch Kalshi Data

fill this out later


### TO DO

1. Missing 6 regular season games; both teams designated 'AWAY' beacuse they wer playing a different country.

    a. Dropping them for now but can easily add them later if we want.

3. Upload Kalshi functions

4. Fetch Kalshi data

